In [1]:
import torch
import torch.nn as nn
import numpy as np 
import matplotlib.pyplot as plt
import sys 
import os
import glob

In [2]:
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/transforms")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/configs")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/plots")

In [3]:
from seg_recon_vit3d_overlap import *
from seg_recon_vit3d_2 import *
from utils.read_yaml import read_yaml
from utils.model_select import model_select

In [4]:
base_path = "/media/ana-caznok/SSD-08/recon-segment/"
config = read_yaml(base_path + 'configs/test_new-vit.yaml')
model,load = model_select(config)

RecDecoder(
  (decoder): Sequential(
    (0): Conv2d(65, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 63, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (4): BatchNorm2d(63, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): Conv2d(63, 62, kernel_size=(5, 5), stride=(3, 3), padding=(1, 1))
    (7): BatchNorm2d(62, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(62, 61, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (10): BatchNorm2d(61, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU(inplace=True)
    (12): Conv2d(61, 61, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
  )
)
Looking for checkpoint in models/test_new-vit.pth. Exact path only: True.

In [5]:
def get_output_shape(transform_class, transform_params, input_shape):
    """
    Computes the output shape of a PyTorch nn transform given the transform class, 
    its parameters, and the input tensor shape.

    Parameters:
    - transform_class (torch.nn.Module): The class of the PyTorch transform (e.g., nn.Conv2d).
    - transform_params (dict): Parameters required to instantiate the transform.
    - input_shape (tuple): The shape of the input tensor (e.g., (1, 3, 224, 224)).

    Returns:
    - tuple: Output tensor shape after applying the transform.
    """

    # Instantiate the transform using the provided class and parameters
    transform = transform_class(**transform_params)

    # Create a dummy input tensor with the specified input shape
    dummy_input = torch.randn(*input_shape)

    # Apply the transform to the dummy input without tracking gradients
    with torch.no_grad():
        output = transform(dummy_input)

    # Return the output shape as a tuple
    return tuple(output.shape)


In [6]:
model.encoder

Encoder_ViT(
  (to_patch_embedding): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=32, p2=32)
    (1): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=4096, out_features=768, bias=True)
    (3): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (dropout): Dropout(p=0, inplace=False)
  (transformer): Transformer(
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (layers): ModuleList(
      (0-5): 6 x ModuleList(
        (0): Attention(
          (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attend): Softmax(dim=-1)
          (dropout): Dropout(p=0, inplace=False)
          (to_qkv): Linear(in_features=768, out_features=2340, bias=False)
          (to_out): Sequential(
            (0): Linear(in_features=780, out_features=768, bias=True)
            (1): Dropout(p=0, inplace=False)
          )
        )
        (1): FeedForward(
          (net): Sequential(
          

In [7]:
B, C, Y, X = 1, 4, 256, 256
E = 768
img = torch.randn(*(B,C,Y,X))

In [ ]:
x = model.encoder(img)

In [ ]:
x.shape

In [ ]:
out = model.decoder(x)

In [ ]:
out.shape

In [ ]:
def get_output_shape(transform_class, transform_params, input_shape):
    """
    Computes the output shape of a PyTorch nn transform given the transform class, 
    its parameters, and the input tensor shape.

    Parameters:
    - transform_class (torch.nn.Module): The class of the PyTorch transform (e.g., nn.Conv2d).
    - transform_params (dict): Parameters required to instantiate the transform.
    - input_shape (tuple): The shape of the input tensor (e.g., (1, 3, 224, 224)).

    Returns:
    - tuple: Output tensor shape after applying the transform.
    """

    # Instantiate the transform using the provided class and parameters
    transform = transform_class(**transform_params)

    # Create a dummy input tensor with the specified input shape
    dummy_input = torch.randn(*input_shape)

    # Apply the transform to the dummy input without tracking gradients
    with torch.no_grad():
        output = transform(dummy_input)

    # Return the output shape as a tuple
    return tuple(output.shape)


In [ ]:
b, t, e1, e2 = x.shape

In [ ]:
256*256*61

In [ ]:
b, t, e1, e2 = x.shape
decoder_input_shape = (b,t,e1,e2)
my_shapes = []
conv_output_shape = get_output_shape(nn.Conv2d,
                        {"in_channels":t,
                        "out_channels":t-1,
                        "kernel_size":3,
                        "stride":1,
                        "padding":1 }, decoder_input_shape)
print(conv_output_shape)
my_shapes.append(conv_output_shape)
t = t-1

In [ ]:
conv_output_shape = get_output_shape(nn.Conv2d,
                        {"in_channels":t,
                        "out_channels":t-1,
                        "kernel_size":3,
                        "stride":2,
                        "padding":1 }, conv_output_shape)
print(conv_output_shape)
my_shapes.append(conv_output_shape)
t=t-1

In [ ]:
conv_output_shape = get_output_shape(nn.Conv2d,
                        {"in_channels":t,
                        "out_channels":t,
                        "kernel_size":5,
                        "stride":3,
                        "padding":1 }, conv_output_shape)
print(conv_output_shape)
my_shapes.append(conv_output_shape)

In [ ]:
conv_output_shape = get_output_shape(nn.ConvTranspose2d,
                        {"in_channels":t,
                        "out_channels":t-1,
                        "kernel_size":4,
                        "stride":2,
                        "padding":1 }, conv_output_shape)
print(conv_output_shape)
my_shapes.append(conv_output_shape)
t=t-1

In [ ]:
conv_output_shape = get_output_shape(nn.ConvTranspose2d,
                        {"in_channels":t,
                        "out_channels":t-1,
                        "kernel_size":3,
                        "stride":1,
                        "padding":1 }, conv_output_shape)
print(conv_output_shape)
my_shapes.append(conv_output_shape)

In [ ]:
b, t, e1, e2 = x.shape
decoder_input_shape = (b,t,e1,e2)
channels =  [512, 256, 128, 64]

conv_shapes = [decoder_input_shape]

for i in range(len(channels) - 1):
    conv_output_shape = get_output_shape(nn.ConvTranspose2d,
                        {"in_channels":channels[i],
                        "out_channels":channels[i + 1],
                        "kernel_size":4,
                        "stride":2,
                        "padding":1 }, 
                 decoder_input_shape)
    conv_shapes.append(conv_output_shape)
    decoder_input_shape = conv_output_shape

In [ ]:
conv_shapes

In [ ]:
b, t, e = x.shape
decoder_input = x.transpose(1, 2).contiguous().view(B, E, h, w)
b_d, e_d, h, w = decoder_input.shape
decoder_input_shape = (b_d,e_d,h, w)

In [ ]:
decoder_input_shape

In [ ]:
x.transpose(1, 2).contiguous().view(B, E, h, w).shape

In [ ]:
upsample_factor = model.decoder.upsample_factor
upsample_layers = model.decoder.num_upsample_layers 
channels =  model.decoder.dec_channels

In [ ]:
decoder_input_shape = (b_d,e_d,h, w)
conv_shapes = [decoder_input_shape]
for i in range(len(channels)-1): 
  conv_output_shape = get_output_shape(
          nn.ConvTranspose2d,
          {"in_channels": channels[i],
            "out_channels": channels[i+1], 
            "kernel_size": 5,
            "stride": 2,
            "padding": 1},
          decoder_input_shape
      )
  conv_shapes.append(conv_output_shape)
  decoder_input_shape = conv_output_shape
#print("Output shape:", conv_output_shape)

In [ ]:
conv_output_shape = get_output_shape(
                    nn.Conv2d,
                    {"in_channels": channels[-1],
                        "out_channels": 61, 
                        "kernel_size": 2,
                        "padding": 1},
                    decoder_input_shape) 
conv_shapes.append(conv_output_shape)

In [ ]:
#kernel size: ou 5 ou 6

In [ ]:
conv_shapes

In [ ]:
256 + 256/2